# Image Representation with Neural Fields

Train neural field models to represent a single image as a continuous function
mapping `(x, y)` coordinates to RGB values.

## Setup

In [ ]:
import math

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch

import neurofield as nf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Load Image

In [ ]:
IMAGE_PATH = "../data/image/kodim17.png"


img = Image.open(IMAGE_PATH)
W, H = img.size
print(f"Image size: {H} x {W}")
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Original")
plt.show()

## Model Configurations

Every neural field has about 200k parameters, so results compare architectures at equal capacity.

In [ ]:
CONFIGS = [
    {
        "name": "RFF",
        "class": nf.RFF,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 3,
            "num_frequencies": 128,
            "sigma": 8.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "PE-MLP",
        "class": nf.PEMLP,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "num_frequencies": 10,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "MFN",
        "class": nf.MFN,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "input_scale": 1024.0,
            "weight_scale": 1e-2,
            "alpha": 1.5,
            "beta": 1.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "SIREN",
        "class": nf.SIREN,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "omega": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,
    },
    {
        "name": "Gauss",
        "class": nf.Gauss,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "scale": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 3e-3,
    },
    {
        "name": "WIRE",
        "class": nf.RealWIRE,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "omega": 20.0,
            "scale": 20.0,
            "output_activation": torch.tanh,
        },
        "lr": 3e-3,
    },
    {
        "name": "FINER",
        "class": nf.FINER,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "omega": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,
    },
    {
        "name": "Instant-NGP",
        "class": nf.InstantNGP,
        "kwargs": {
            "num_levels": 18,
            "log2_hashmap_size": 13,
            "max_resolution": max(H, W),
            "hidden_features": 64,
            "hidden_layers": 1,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "TensoRF",
        "class": nf.TensoRF,
        "kwargs": {
            "rank": 176,
            "resolution": 256,
            "mode": "cp",
            "hidden_features": 256,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "GA-Planes",
        "class": nf.GAPlanes,
        "kwargs": {
            "features": 25,
            "resolution": (256, 64),
            "hidden_features": 256,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "FUTON-cosine",
        "class": nf.FUTON,
        "kwargs": {
            "basis": (
                "cosine",
                {"num_components": (H // 2, W // 2), "grid_size": (H, W)},
            ),
            "combiner": ("cp", {"rank": 224}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 0.1,
    },
    {
        "name": "FUTON-sinc",
        "class": nf.FUTON,
        "kwargs": {
            "basis": (
                "sinc",
                {"num_components": (H // 2, W // 2), "grid_size": (H, W)},
            ),
            "combiner": ("cp", {"rank": 224}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "FUTON-triangle",
        "class": nf.FUTON,
        "kwargs": {
            "basis": ("triangle", {"num_components": (H // 2, W // 2)}),
            "combiner": ("cp", {"rank": 224}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "FUTON-lanczos",
        "class": nf.FUTON,
        "kwargs": {
            "basis": ("lanczos", {"num_components": (H // 2, W // 2), "radius": 3}),
            "combiner": ("cp", {"rank": 224}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
]

## Train Models

In [ ]:
results = []

for config in CONFIGS:
    torch.manual_seed(0)

    train_dataset = nf.ImageCoordinateDataset(IMAGE_PATH, subsample=0.1)
    eval_dataset = nf.ImageCoordinateDataset(IMAGE_PATH)

    model = config["class"](in_features=2, out_features=3, **config["kwargs"])
    num_params = nf.count_parameters(model, trainable_only=False)

    print(f"\n--- {config['name']} ({num_params:,} params) ---")
    res = nf.train(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        lr=config["lr"],
        num_epochs=2000,
        metrics={
            "psnr": nf.psnr,
            "ssim": nf.ssim,
            "ms_ssim": nf.ms_ssim,
            "lpips": nf.lpips,
        },
        log_interval=100,
        eval_interval=100,
        device=device,
    )

    output = nf.chunked_inference(model, eval_dataset.input, device=device)
    recon = eval_dataset.postprocess(output.cpu())

    res["config"]["model"] = config["name"]
    res["reconstructed"] = recon
    results.append(res)

    del model, recon, res["model_dict"]
    torch.cuda.empty_cache()

## Collect Results

In [ ]:
records = []
for res in results:
    for entry in res["history"]:
        if "eval" not in entry:
            continue

        records.append(
            {
                "Model": res["config"]["model"],
                "# Params (k)": res["config"]["num_params"] / 1000,
                "Iteration": entry["epoch"],
                "Time (s)": entry["elapsed"],
                "Speed (img/s)": 1.0 / entry["eval"]["duration"],
                "PSNR (dB)": entry["eval"]["psnr"],
                "SSIM": entry["eval"]["ssim"],
                "LPIPS": entry["eval"]["lpips"],
            }
        )


records = pd.DataFrame(records)

## Convergence Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.lineplot(
    data=records,
    x="Iteration",
    y="PSNR (dB)",
    hue="Model",
    style="Model",
    markers=True,
    ax=axes[0],
)
axes[0].set_ylim(27, 41)
axes[0].grid(alpha=0.3)
axes[0].legend(loc="lower right")

sns.lineplot(
    data=records,
    x="Time (s)",
    y="PSNR (dB)",
    hue="Model",
    style="Model",
    markers=True,
    ax=axes[1],
)
axes[1].set_xlim(0, 30)
axes[1].set_ylim(27, 41)
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower right")

plt.show()

## Results Summary

In [ ]:
summary = (
    records.groupby("Model")
    .agg(
        {
            "# Params (k)": "first",
            "Iteration": "last",
            "Time (s)": "last",
            "Speed (img/s)": "mean",
            "PSNR (dB)": "last",
            "SSIM": "last",
            "LPIPS": "last",
        }
    )
    .sort_values("PSNR (dB)", ascending=False)
    .reset_index()
)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(36, 9))

for ax, metric in zip(axes, ["PSNR (dB)", "SSIM", "Speed (img/s)", "# Params (k)"]):
    summary_sorted = summary.sort_values(metric, ascending=True)
    ax.barh(summary_sorted["Model"], summary_sorted[metric])
    ax.set_xlabel(metric)
    ax.grid(axis="x", alpha=0.3)

plt.show()

## Visual Comparison

In [ ]:
psnr_lookup = summary.set_index("Model")["PSNR (dB)"]
sorted_results = sorted(
    results, key=lambda r: psnr_lookup[r["config"]["model"]], reverse=True
)

items = [("Original", eval_dataset.original)] + [
    (res["config"]["model"], res["reconstructed"]) for res in sorted_results
]
num_cols = math.ceil(len(items) / 2)
fig, axes = plt.subplots(2, num_cols, figsize=(2 * num_cols, 6))

for idx, (name, img) in enumerate(items):
    ax = axes[idx // num_cols, idx % num_cols]
    ax.imshow(img.permute(1, 2, 0))
    title = name if name == "Original" else f"\n{name}\nPSNR: {psnr_lookup[name]:.1f}dB"
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()